In [1]:
import numpy as np
import matplotlib.pyplot as plt
import vrplib
import copy
from pathlib import Path
from typing import Tuple, Dict, List, Set, Union

In [2]:
from two_regret import *

In [3]:
RouteDict = Dict[str, Union[List[int], float]]
Coords = Union[Dict[int, Tuple[float, float]], np.ndarray, List[List[float]]]
PlotMetadata = Dict[str, Union[str, float]]

In [4]:
def unpacking_data(data: Dict) -> Tuple[int, float, np.ndarray, np.ndarray]:
    """
    Extract data from an instance and calculate the Euclidean distance matrix.

    Parameters
    ----------
    data: dict

    Return
    -------
    N:  int
        Number of node
    K:  int
        Number of vehicles
    Q:  float
        Maximum load capacity
    d:  np.ndarray
        Demand vector
    D:  np.ndarray
        Distance matrix (N x N)
    """
    
    N = data['dimension']
    K = int(data['name'].split('k')[1])
    Q = data['capacity']
    d = data['demand']
    node_coord = data['node_coord']
    D = np.zeros((N, N))

    for i in range(N):
        for j in range(i + 1, N):
            dist = np.hypot(node_coord[j][0] - node_coord[i][0], node_coord[j][1] - node_coord[i][1]) 
            D[j, i] = D[i, j] = dist

    return N, K, Q, d, D, node_coord

In [5]:
def wo_regret(N: int, K: int, Q: float, d: np.ndarray, D: np.ndarray) -> List[RouteDict]:
    """
    Builds an initial solution for the CVRP using the 2-Regret heuristic.

    Parameters
    ------------
    N:  int
        Number of nodes.
    K:  int  
        Number of vehicles.
    Q:  float
        Maximum load capacity of each vehicle.
    d:  np.ndarray
        Demand vector.
    D:  np.ndarray
        Distance matrix.

    Return:
    ---------
    routes: list[dict]
            A list of dictionaries representing the routes and their current loads.
    """
    
    routes = [{"route": [0, 0], "capacity": 0.0} for _ in range(K)]
    R = {0}  
    
    while len(R) < N:
        node_data = {} 

        for node in range(1, N):
            if node in R:
                continue

            best_by_route = {}

            for k in range(K):
                
                if routes[k]["capacity"] + d[node] <= Q:
                    route = routes[k]["route"]
                    
                    best_cost_in_k = float('inf')
                    best_pos_in_k = -1
                    
                    for i in range(len(route) - 1):
                        a, b = route[i], route[i + 1]
                        cost = D[a][node] + D[node][b] - D[a][b]
                        
                        if cost < best_cost_in_k:
                            best_cost_in_k = cost
                            best_pos_in_k = i + 1
                    
                    if best_pos_in_k != -1:
                        best_by_route[k] = {
                            "cost": best_cost_in_k, 
                            "route_idx": k, 
                            "pos": best_pos_in_k
                        }

            if not best_by_route:
                continue

            sorted_route_options = sorted(best_by_route.values(), key=lambda x: x["cost"])

            best_opt = sorted_route_options[0]

            if len(sorted_route_options) > 1:
                regret = sorted_route_options[1]["cost"] - best_opt["cost"]
            else:
                regret = 1e9 + best_opt["cost"] 

            node_data[node] = {"regret": regret, "best_info": best_opt}

        if not node_data:
            print("Warning: Não foi possível inserir todos os nós com as restrições atuais.")
            break

        best_node = max(node_data, key=lambda n: node_data[n]["regret"])
        
        info = node_data[best_node]["best_info"]
        k_idx = info["route_idx"]
        pos = info["pos"]
        
        routes[k_idx]["route"].insert(pos, int(best_node))
        routes[k_idx]["capacity"] += float(d[best_node])
        R.add(best_node)

    return routes

In [6]:
def cost_per_route(route: List[int], D: np.ndarray) -> float:
    """
    Calculate the cost of a route based on the distance matrix.

    Parameters
    ------------
    route:  list
            List of nodes.
    D:      np.ndarray
            Distance matrix        
   
    Return:
    ---------
    cost:   floatsolution_2regret.png
            The sum of the distances of all the nodes of the route.
    """

    cost = sum(D[i][j] for i, j in zip(route[:-1], route[1:]))

    return cost

def total_cost(routes: List[RouteDict], D: np.ndarray) -> float:
    """
    Calculate the total cost of all routes

    Parameters
    ------------
    route:  list[dict]
            A list of dictionaries representing the routes and their current loads.
    D:      np.ndarray
            Distance matrix.    
   
    Return:
    ---------
    cost:   float
            The sum of the distances of all the nodes of the route.
    """

    cost = sum(cost_per_route(r["route"], D) for r in routes)

    return cost

In [7]:
def swap_1(routes: List[RouteDict], D: np.ndarray, d: List[float], Q: float) -> List[RouteDict]:
    """
    Performs the First-Improvement Inter-Route Swap
    
    Parameters:
    ------------
    routes: list[dict]
        A list of dictionaries representing the routes and their current loads.
    D:  np.ndarray
        Distance matrix.
    d:  np.ndarray
        Demand vector.
    Q:  float
        Maximum load capacity of each vehicle.

    Return:
    ---------
    routes: list[dict]
            A list of dictionaries representing the routes and their current loads.
    """

    n_routes = len(routes)
    status = False
        
    for i in range(n_routes):
        for j in range(i + 1, n_routes):

            route_a = routes[i]
            route_b = routes[j]
            
            for idx_a in range(1, len(route_a["route"]) - 1):
                for idx_b in range(1, len(route_b["route"]) - 1):
                    
                    node_a = route_a["route"][idx_a]
                    node_b = route_b["route"][idx_b]

                    new_capacity_a = route_a["capacity"] - d[node_a] + d[node_b]
                    new_capacity_b = route_b["capacity"] - d[node_b] + d[node_a]

                    if new_capacity_a <= Q and new_capacity_b <= Q:
                        
                        current_total_cost = (cost_per_route(route_a["route"], D) + cost_per_route(route_b["route"], D))
                        
                        temp_route_i = list(route_a["route"])
                        temp_route_j = list(route_b["route"])
                        temp_route_i[idx_a] = node_b
                        temp_route_j[idx_b] = node_a
                        
                        new_total_cost = (cost_per_route(temp_route_i, D) + cost_per_route(temp_route_j, D))

                        if new_total_cost < current_total_cost:

                            route_a["route"] = temp_route_i
                            route_b["route"] = temp_route_j
                            route_a["capacity"] = new_capacity_a
                            route_b["capacity"] = new_capacity_b
                            status = True

                            return routes, status

    return routes, status

In [8]:
def local_search(routes: List[RouteDict], D: np.ndarray, d: List[float], Q: float):

    routes_ = routes
    success = True

    while success:

        routes_, status = swap_1(routes_, D, d, Q)

        if not status:
            success = False
            
    return routes_

In [ ]:
def plot_nodes(instances: list[dict], figsize: tuple[float, float] = (15, 10), show: bool = True, save_path: str | Path | None = None):

    n = len(instances)
    ncols = n
    nrows = 1

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_2d(axes).flatten()

    for i, inst in enumerate(instances):
        ax = axes[i]
        coords = inst['coords']
        demands = inst['demands']
        depot = 0

        is_customer = np.arange(inst['dimension']) != depot
        dmax = max(demands.max(), 1)
        sizes = 30 + (demands[is_customer] / dmax) * 220 
        ax.scatter(
            coords[is_customer, 0],
            coords[is_customer, 1],
            s=sizes,
            c="steelblue",
            alpha=0.75,
            edgecolors="navy",
            linewidth=0.6,
            label="Clientes",
        )

        ax.scatter(
            coords[depot, 0],
            coords[depot, 1],
            s=320,
            c="crimson",
            marker="s",
            edgecolors="black",
            linewidth=1.5,
            label="Escola (depósito)",
            zorder=5,
        )
        ax.annotate(
            "Escola",
            coords[depot],
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
            color="crimson",
        )

        opt_str = f", ótimo CVRP={inst['optimal_cvrp']:.2f}" if inst['optimal_cvrp'] else ""
        ax.set_title(
            f"{inst['name']}\n"
            f"n={inst['n_customers']}, K={inst['n_trucks']}, Q={inst['capacity']}{opt_str}\n"
            f"demanda total = {inst['total_demand']}",
            fontsize=10,
        )
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.grid(True, alpha=0.3)
        ax.set_aspect("equal", adjustable="box")
        if i == 0:
            ax.legend(loc="best", fontsize=8, borderpad=1.5, labelspacing=1.5, handlelength=2.0)
    
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Figura salva em: {save_path}")

    if show:
        plt.show()
    else:
        return fig

In [10]:
def plot_routes(instances: list[dict], routes: list[RouteDict], figsize: tuple[float, float] = (15, 10), show: bool = True, save_path: str | Path | None = None):

    n = len(instances)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_2d(axes).flatten()

    for i, inst in enumerate(instances):
        ax = axes[i]
        coords = inst['coords']
        demands = inst['demands']
        depot = 0

        is_customer = np.arange(inst['dimension']) != depot
        dmax = max(demands.max(), 1)
        sizes = 30 + (demands[is_customer] / dmax) * 220 
        ax.scatter(
            coords[is_customer, 0],
            coords[is_customer, 1],
            s=sizes,
            c="steelblue",
            alpha=0.75,
            edgecolors="navy",
            linewidth=0.6,
            label="Clientes",
        )

        ax.scatter(
            coords[depot, 0],
            coords[depot, 1],
            s=320,
            c="crimson",
            marker="s",
            edgecolors="black",
            linewidth=1.5,
            label="Escola (depósito)",
            zorder=5,
        )
        ax.annotate(
            "Escola",
            coords[depot],
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
            color="crimson",
        )

        routes_inst = routes[i]

        if routes_inst is not None:

            colors = plt.cm.tab10.colors

            for r_idx, r in enumerate(routes_inst):

                route = r["route"]

                rx = [coords[i][0] for i in route]
                ry = [coords[i][1] for i in route]

                ax.plot(
                    rx,
                    ry,
                    color=colors[r_idx % len(colors)],
                    linewidth=2,
                    alpha=0.9,
                    label=f"Rota {r_idx+1}",
                )

        opt_str = inst['optimal_cvrp'] if inst['optimal_cvrp'] else ""
        cost = inst['cost']
        gap_opt = float('inf') if cost == 0 else abs(cost - opt_str) / abs(cost) * 100

        ax.set_title(
            f"{inst['name']}\n"
            f"custo={cost:.2f}, ótimo={opt_str} (gap {gap_opt:.2f}%)\n",
            fontsize=10,
        )
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.grid(True, alpha=0.3)
        ax.set_aspect("equal", adjustable="box")
        
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Figura salva em: {save_path}")

    if show:
        plt.show()
    else:
        return fig

In [11]:
path_files = 'instancias/'
files_name = ['A-n32-k5.vrp', 'A-n33-k5.vrp', 'A-n33-k6.vrp', 'A-n34-k5.vrp', 'A-n36-k5.vrp', 'toy_problem.vrp']
opt_val = [784, 661, 742, 778, 779, 401.53]

In [12]:
def two_regret(N: int, K: int, Q: float, d: np.ndarray, D: np.ndarray) -> List[List[int]]:
    """
    Builds an initial solution for the CVRP using the 2-Regret heuristic.

    Parameters
    ------------
    N:  int
        Number of nodes.
    K:  int  
        Number of vehicles.
    Q:  float
        Maximum load capacity of each vehicle.
    d:  np.ndarray
        Demand vector.
    D:  np.ndarray
        Distance matrix.

    Return:
    ---------
    routes: list[dict]
            A list of dictionaries representing the routes and their current loads.
    """
    
    routes = [[0, 0] for _ in range(K)]
    capacity = N * [0.0] 
    R = {0} # Visited nodes  
    
    while len(R) < N:

        node_data = {}

        for node in range(1, N):
            if node in R:
                continue

            best_by_route = {}

            for k in range(K):
                
                if capacity[k] + d[node] <= Q:
                    route = routes[k]
                    
                    best_cost_in_k = float('inf')
                    best_pos_in_k = -1
                    
                    for i in range(len(route) - 1):
                        a, b = route[i], route[i + 1]
                        cost = D[a][node] + D[node][b] - D[a][b]
                        
                        if cost < best_cost_in_k:
                            best_cost_in_k = cost
                            best_pos_in_k = i + 1
                    
                    if best_pos_in_k != -1:
                        best_by_route[k] = {
                            "cost": best_cost_in_k, 
                            "route_idx": k, 
                            "pos": best_pos_in_k
                        }

            if not best_by_route:
                continue

            sorted_route_options = sorted(best_by_route.values(), key=lambda x: x["cost"])

            best_opt = sorted_route_options[0]

            if len(sorted_route_options) > 1:
                regret = sorted_route_options[1]["cost"] - best_opt["cost"]
            else:
                regret = 1e9 + best_opt["cost"] 

            node_data[node] = {"regret": regret, "best_info": best_opt}

        if not node_data:
            print("Warning: It was not possible to insert all nodes with the current constraints.")
            return [[]] # Return empty route

        best_node = max(node_data, key=lambda n: node_data[n]["regret"])
        
        info = node_data[best_node]["best_info"]
        k_idx = info["route_idx"]
        pos = info["pos"]
        
        routes[k_idx].insert(pos, int(best_node))
        capacity[k_idx] += float(d[best_node])
        R.add(best_node)

    return routes

In [13]:
# Toy problem

N, K, Q, d, D, node_coord = unpacking_data(vrplib.read_instance(path_files + files_name[1]))
routes = wo_regret(N, K, Q, d, D)
total_cost_routes = total_cost(routes, D)

In [15]:
two_regret_insertion(N, K, Q, d, D)

[[0, 19, 14, 21, 1, 31, 29, 3, 16, 26, 32, 0],
 [0, 25, 5, 7, 8, 13, 2, 0],
 [0, 17, 10, 30, 27, 4, 0],
 [0, 24, 6, 11, 18, 28, 23, 9, 0],
 [0, 20, 12, 15, 22, 0]]

In [14]:
routes

[{'route': [0, 19, 14, 21, 1, 31, 29, 3, 16, 26, 32, 0], 'capacity': 99.0},
 {'route': [0, 25, 5, 7, 8, 13, 2, 0], 'capacity': 97.0},
 {'route': [0, 17, 10, 30, 27, 4, 0], 'capacity': 100.0},
 {'route': [0, 24, 6, 11, 18, 28, 23, 9, 0], 'capacity': 96.0},
 {'route': [0, 20, 12, 15, 22, 0], 'capacity': 54.0}]

In [ ]:
two_regret(N, 1, Q, d, D)

In [ ]:
data_nodes = {}
data_nodes['coords'] = node_coord
data_nodes['demands'] = d
data_nodes['dimension'] = N
data_nodes['optimal_cvrp'] = opt_val[-1]
data_nodes['name'] = files_name[-1]
data_nodes['n_customers'] = N - 1
data_nodes['n_trucks'] = K
data_nodes['capacity'] = Q
data_nodes['total_demand'] = np.sum(d)

In [ ]:
plot_nodes([data_nodes], save_path='toy_problem.pdf')

In [ ]:
data_nodes['cost'] = total_cost_routes
plot_routes([data_nodes], [routes], save_path='toy_problem_2_regret.pdf')

In [ ]:
augerat_initial_data = []
augerat_initial_routes = []
augerat_local_data = []
augerat_local_routes = []

for idx, file in enumerate(files_name[:-1]):
    
    N, K, Q, d, D, node_coord = unpacking_data(vrplib.read_instance(path_files + file))
    data_nodes = {}
    data_nodes['coords'] = node_coord
    data_nodes['demands'] = d
    data_nodes['dimension'] = N
    data_nodes['optimal_cvrp'] = opt_val[idx]
    data_nodes['name'] = file
    data_nodes['cost'] = 0.0

    # Inital solution
    routes = two_regret(N, K, Q, d, D)
    total_cost_routes_initial = total_cost(routes, D)

    data_nodes["cost"] = total_cost_routes_initial
    augerat_initial_data.append(data_nodes)
    augerat_initial_routes.append(routes)
    
    # Local search
    local_search_ = local_search(routes, D, d, Q)
    total_cost_routes_local = total_cost(local_search_, D)

    data_nodes_ = data_nodes.copy()
    data_nodes_['cost'] = total_cost_routes_local
    augerat_local_data.append(data_nodes_)
    augerat_local_routes.append(local_search_)   

In [ ]:
plot_routes(augerat_initial_data, augerat_initial_routes, save_path='augerat_initial.pdf')

In [ ]:
plot_routes(augerat_local_data, augerat_local_routes, save_path='augerat_local.pdf')